# gras — a short Python tour

`gras` searches neural architectures while their weights train: a population of
candidate nets shares one seeded stream, is scored every clock, and the weak ones
are culled as they fall behind.

Three independent examples: a supervised race on MNIST, a reinforcement-learning
race on CartPole, and an in-memory array race where loss and fitness differ. Each
code cell imports what it needs and can run on its own.

**Before running:** `gras` must be installed in the kernel's environment. It links
libtorch rather than bundling it, so a compatible libtorch must be available
(`LIBTORCH_PATH` or the loader's path). `data/blobs` must exist; the MNIST cell
downloads and caches its dataset on first run; CartPole needs `gymnasium`.

The Python surface aims to cover the Rust feature set, but Rust remains the more
expressive door when you need custom losses, fitness functions, or finer control.

In [ ]:
# ── Supervised race: MNIST ───────────────────────────────────────────────────
# Parity recipe for seed 13827759280116606227. The Rust `mnist` example now
# defaults its hand-rolled trainer's challenge_prob to 0.0 and uses the same
# batch/call-index path as `TabularTrainer`, so the two runs write the same
# history. First run downloads the four IDX files into data/mnist/gz and
# converts them to the crate's GRA1 tensors; later runs use the cache.
import gzip
import json
import struct
import sys
import tempfile
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

import gras
from gras import RaceConfig, RaceEngine

MNIST_ROOT = Path("data/mnist")
MNIST_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"
MNIST_FILES = (
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
)


def read_idx(path: Path) -> np.ndarray:
    """Read one uncompressed IDX payload and return it with its declared shape."""
    with gzip.open(path, "rb") as f:
        # IDX magic 0x00000803 / 0x00000801: low byte is the dimension count.
        magic = struct.unpack(">I", f.read(4))[0]
        ndim = magic & 0xFF
        shape = struct.unpack(f">{ndim}I", f.read(4 * ndim))
        data = np.frombuffer(f.read(), dtype=np.uint8)
    assert magic in (2049, 2051), f"unexpected IDX magic in {path}"
    return data.reshape(shape)


def write_tensor(path: Path, array: np.ndarray) -> None:
    """Write the crate's GRA1 format: magic, float32 tag, shape, raw little-endian f32."""
    array = np.ascontiguousarray(array, dtype=np.float32)
    with path.open("wb") as f:
        f.write(b"GRA1\x00")
        f.write(struct.pack("<Q", array.ndim))
        f.write(struct.pack(f"<{array.ndim}Q", *array.shape))
        f.write(array.tobytes())


def cache_mnist() -> None:
    gz_dir = MNIST_ROOT / "gz"
    gz_dir.mkdir(parents=True, exist_ok=True)
    for name in MNIST_FILES:
        dest = gz_dir / name
        if not dest.is_file():
            urlretrieve(MNIST_URL + name, dest)
            print(f"downloaded {dest}", flush=True)

    splits = {
        "train": ("train-images-idx3-ubyte.gz", "train-labels-idx1-ubyte.gz"),
        "test": ("t10k-images-idx3-ubyte.gz", "t10k-labels-idx1-ubyte.gz"),
    }
    for split, (images_name, labels_name) in splits.items():
        target = MNIST_ROOT / split
        if (target / "inputs.bin").is_file():
            continue
        images = read_idx(gz_dir / images_name).reshape(-1, 28 * 28)
        labels = read_idx(gz_dir / labels_name)
        x = images.astype(np.float32) / 255.0
        y = np.eye(10, dtype=np.float32)[labels]
        target.mkdir(parents=True, exist_ok=True)
        write_tensor(target / "inputs.bin", x)
        write_tensor(target / "targets.bin", y)
        (target / "meta.json").write_text(json.dumps({
            "inputs_shape": list(x.shape),
            "targets_shape": list(y.shape),
        }, indent=2))
        print(f"wrote {target} | {x.shape[0]} rows x {x.shape[1]} features", flush=True)


if not all((MNIST_ROOT / s / "inputs.bin").is_file() for s in ("train", "test")):
    cache_mnist()
else:
    print(f"using cached MNIST at {MNIST_ROOT}", flush=True)
MNIST = MNIST_ROOT  # explicit train/test layout: same parent as Rust --data-dir

run_dir = Path(tempfile.mkdtemp(prefix="gras-mnist-"))
cfg = RaceConfig(
    # Only options that differ from RaceConfig()'s aligned defaults.
    stop_plateau=(0.85, 40),
    checkpoint_every=50,
    train_merit_p_min=0.15,
    train_merit_p_max=0.85,
    mutation_novelty_min=0.3,
    mutation_novelty_draws=5,
    crossover_share=0.025,
    mutation_share=0.005,
    recovery_same_seed=True,
    dropout_prob=0.2,
    input_reach=(0.05, 0.95),
    worst_save_topology=True,
    run_name="mnist",  # metadata only; drop if you do not need the run label
)
engine = RaceEngine(
    MNIST,
    cfg,
    seed=13827759280116606227,
    run_dir=run_dir,
    loss="label_smoothing_ce(0.1)",  # same smoothed CE as the Rust example
    fitness="accuracy",
    lr=1e-3,
    grad_clip=2.0,
    batch_size=32,
    eval_batch_size=64,
    optimizer="adam",
)

print(f"gras {gras.__version__} | run dir {engine.run_dir}", flush=True)
stop = engine.run()
sys.stdout.flush()
print(f"stop: {stop} after {engine.step_clock} clocks", flush=True)

# One row per net per clock: train_loss minimized, eval_loss held out,
# fitness ranked. accuracy is maximize: higher is better.
history = pd.read_csv(run_dir / "history.csv")
header = json.loads((run_dir / "engine.json").read_text())
print(f"\n{len(history)} readings across {history['hash'].nunique()} nets", flush=True)
print(history.groupby("step")[["train_loss", "eval_loss", "fitness"]].mean().round(3), flush=True)
best = history.loc[history["fitness"].idxmax()]
print(f"\nbest net {best['hash']} at clock {int(best['step'])}: {best['fitness']:.3f}", flush=True)
print(f"recipe: {header['engine_mode']} | {header['input_dim']} -> {header['output_dim']} | loss={header['trainer']['loss']}", flush=True)

In [ ]:
# ── Reinforcement learning: CartPole ────────────────────────────────────────
# Same RaceConfig defaults as the Rust `cartpole` example, with only the
# non-default options below. The trainer is the crate's Reinforce.
import sys
import tempfile
from pathlib import Path

import pandas as pd

import gras
from gras import RaceConfig, RaceEngine

try:
    import gymnasium
except ImportError:
    gymnasium = None

if gymnasium is None:
    print("skipped: pip install gymnasium")
else:
    run_dir = Path(tempfile.mkdtemp(prefix="gras-cartpole-"))
    cfg = RaceConfig(
        # Only options that differ from RaceConfig()'s aligned defaults.
        pop_size=20,
        champion_count=2,
        max_steps=20,
        stop_plateau=(400.0, 0.0),
        checkpoint_every=50,
        drift_prob=0.05,
        mutation_novelty_min=0.3,
        hidden_dim_stride=2,
        hidden_nodes_pool=(2, 20),
        hidden_inputs_pool=(2, 20),
        hidden_outputs_pool=(2, 20),
        train_merit_p_min=0.15,
        train_merit_p_max=0.85,
        crossover_share=0.025,
        mutation_share=0.005,
        recovery_same_seed=True,
        worst_save_topology=True,
        run_name="cartpole",
    )
    engine = RaceEngine.gym(
        gymnasium.make("CartPole-v1"),
        cfg,
        seed=42,
        run_dir=run_dir,
        lr=1e-4,
        grad_clip=2.0,
        matches_per_step=1,
        eval_matches_per_step=5,
    )

    print(f"gras {gras.__version__} from {gras.__file__} | run dir {engine.run_dir}", flush=True)
    stop = engine.run()
    sys.stdout.flush()
    print(f"stop: {stop} after {engine.step_clock} clocks", flush=True)

    history = pd.read_csv(run_dir / "history.csv")
    print(history.groupby("step")["fitness"].agg(["min", "mean", "max"]).round(1), flush=True)
    print(f"\nlongest episode: {history['fitness'].max():.0f} turns", flush=True)


In [ ]:
# ── In-memory arrays: loss and fitness are different ────────────────────────
# from_arrays writes native inputs.bin/targets.bin, then builds the same engine.
# MSE is optimized, MAE is ranked, R² is recorded and never selected on.
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import gras
from gras import RaceConfig, RaceEngine

rng = np.random.default_rng(0)
X = rng.random((512, 8), dtype=np.float32)
w = rng.normal(size=(8, 1)).astype(np.float32)
Y = (X @ w).astype(np.float32)

work = Path(tempfile.mkdtemp(prefix="gras-arrays-"))
data_dir = work / "data"

cfg = RaceConfig(
    pop_size=8,
    champion_count=2,
    max_steps=6,
    checkpoint_every=3,
    log_level="summ",
)
race = RaceEngine.from_arrays(
    X,
    Y,
    data_dir,
    cfg,
    seed=7,
    loss="mse",
    fitness="mae",
    metrics=["r2"],
    lr=2e-3,
    run_dir=work / "run",
)

print(f"gras {gras.__version__} | run dir {race.run_dir}", flush=True)
stop = race.run()
sys.stdout.flush()
print(f"stop: {stop} after {race.step_clock} clocks", flush=True)

history = pd.read_csv(Path(race.run_dir) / "history.csv")
print(f"\nwrote: {sorted(p.name for p in data_dir.iterdir())}", flush=True)
print(history.groupby("step")[["train_loss", "fitness", "r2"]].mean().round(3), flush=True)
print("\ntrain_loss = minimized MSE | fitness = ranking MAE | r2 = recorded only", flush=True)

## Where next

- `python/README.md` — package surface, datasets, artifacts and limits.
- `python/tests/test_determinism.py` — the executable contract, including every
  menu entry.
- `gras.menus()` — every keyword and the names it accepts.
- `gras.feature_splits()` — `full`, `random`, `exclusive`, `chunks`.
- Rust examples: `examples/mnist.rs` and `examples/cartpole.rs`.

Run directories here are temporary. Pass `run_dir=Path("results/mine")` to keep
one.